## Imports & Setup
Importing core libraries for data manipulation, visualisation, sentiment analysis (VADER and BERT), semantic embeddings, and classification modelling.

In [10]:
import pandas as pd
import numpy as np
import altair as alt
import os
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from transformers import pipeline
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv()

True

--- 

## Data Loading
Loading the raw Marvel Comics dataset containing ~35,000 entries with 12 features including comic descriptions, ratings, imprints, and publication metadata.

In [11]:
# Load data
df = pd.read_csv("Marvel_Comics.csv")
print(f"Raw shape: {df.shape}")
df.head()

Raw shape: (34992, 12)


,comic_name,active_years,issue_title,publish_date,issue_description,penciler,writer,cover_artist,Imprint,Format,Rating,Price
0,A Year of Marvels: April Infinite Comic (2016),(2016),A Year of Marvels: April Infinite Comic (2016) #1,"April 01, 2016",The Infinite Comic that will have everyone tal...,Yves Bigerel,Yves Bigerel,Jamal Campbell,Marvel Universe,Infinite Comic,Rated T+,Free
1,A Year of Marvels: August Infinite Comic (2016),(2016),A Year of Marvels: August Infinite Comic (2016...,"August 10, 2016","It’s August, and Nick Fury is just in time to ...",Jamal Campbell,"Chris Sims, Chad Bowers",NaN,Marvel Universe,Infinite Comic,NaN,Free
2,A Year of Marvels: February Infinite Comic (2016),(2016),A Year of Marvels: February Infinite Comic (20...,"February 10, 2016",Join us in a brand new Marvel comics adventure...,"Danilo S. Beyruth, M Mast",Ryan North,NaN,Marvel Universe,Infinite Comic,Rated T+,Free
3,A Year of Marvels: July Infinite Comic (2016),(2016),A Year of Marvels: July Infinite Comic (2016) #1,"June 29, 2016",Celebrating the Fourth of July is complicated ...,Juanan Ramirez,Chuck Wendig,Jamal Campbell,Marvel Universe,Infinite Comic,NaN,Free
4,A Year of Marvels: June Infinite Comic (2016),(2016),A Year of Marvels: June Infinite Comic (2016) #1,"June 15, 2016",Sam Alexander’s finding it hard to cope with t...,Diego Olortegui,Paul Allor,Jamal Campbell,Marvel Universe,Infinite Comic,NaN,Free


---

## Data Cleaning & Preprocessing
Dropping rows without descriptions, stripping whitespace artefacts from rating labels, and consolidating 36 inconsistent rating variants into five standardised categories (All Ages, Teen, Teen+, Parental Advisory, Mature/Explicit). A modelling subset of ~11,000 labelled rows is created for supervised learning, while the full 30,000-row dataset is retained for exploratory analysis.

In [12]:
# Drop rows without descriptions (we need text for NLP)
df = df.dropna(subset=['issue_description'])
print(f"After dropping null descriptions: {df.shape}")

# Clean ratings - consolidate duplicates into standard groups
rating_map = {
    'Rated T': 'Teen', 'T': 'Teen', 'RATED T': 'Teen',
    'Rated T+': 'Teen+', 'T+': 'Teen+', 'RATED T+': 'Teen+',
    'All Ages': 'All Ages', 'A': 'All Ages', 'ALL AGES': 'All Ages',
    'Parental Advisory': 'Parental Advisory', 'PARENTAL ADVISORY': 'Parental Advisory',
    'Marvel Psr': 'Marvel PSR', 'MARVEL PSR': 'Marvel PSR',
    'Max': 'MAX', 'MAX': 'MAX', 'Explicit Content': 'MAX',
    'EXPLICIT CONTENT': 'MAX',
}

df['rating_clean'] = df['Rating'].map(rating_map).fillna('Other')

# Clean imprint casing
df['imprint_clean'] = df['Imprint'].str.strip().str.title().fillna('Unknown')

print(f"\nCleaned rating distribution:")
print(df['rating_clean'].value_counts())

After dropping null descriptions: (30395, 12)

Cleaned rating distribution:
rating_clean
Other    30395
Name: count, dtype: int64


In [13]:
df['Rating'].value_counts(dropna=False)

Rating
NaN                                    18018
 Rated T+                               3183
 T                                      1544
 Rated T                                1392
 Parental Advisory                      1200
 T+                                      934
 All Ages                                881
 A                                       633
 MARVEL PSR                              523
 Marvel Psr                              504
 ALL AGES                                292
 PARENTAL ADVISORY                       207
 RATED T+                                206
 Rated a                                 135
 EXPLICIT CONTENT                        110
 RATED A                                 103
 No Rating                                88
 RATED T                                  84
 PARENTAL ADVISORY/EXPLICIT CONTENT       69
 Parental Advisory/Explicit Content       68
 Mature                                   42
 Explicit Content                         39
 Ra

In [14]:
# Clean ratings - consolidate duplicates into standard groups
rating_map = {
    # Teen+
    'Rated T+': 'Teen+', 'T+': 'Teen+', 'RATED T+': 'Teen+',

    # Teen
    'Rated T': 'Teen', 'T': 'Teen', 'RATED T': 'Teen',

    # All Ages
    'All Ages': 'All Ages', 'A': 'All Ages', 'ALL AGES': 'All Ages',
    'Rated a': 'All Ages', 'RATED A': 'All Ages', 'Rated A': 'All Ages',
    'Ages 10 & Up': 'All Ages',

    # Parental Advisory
    'Parental Advisory': 'Parental Advisory', 'PARENTAL ADVISORY': 'Parental Advisory',
    'PARENTAL ADVISORYSLC': 'Parental Advisory',

    # Mature/Explicit
    'EXPLICIT CONTENT': 'Mature/Explicit', 'Explicit Content': 'Mature/Explicit',
    'PARENTAL ADVISORY/EXPLICIT CONTENT': 'Mature/Explicit',
    'Parental Advisory/Explicit Content': 'Mature/Explicit',
    'Mature': 'Mature/Explicit', 'Max': 'Mature/Explicit',

    # PSR (Pre-existing Standards Rating - older Marvel system)
    'MARVEL PSR': 'Marvel PSR', 'Marvel Psr': 'Marvel PSR',
    'Marvel Psr+': 'Marvel PSR',

    # No rating
    'No Rating': 'No Rating', 'NO RATING': 'No Rating',
}

df['rating_clean'] = df['Rating'].map(rating_map).fillna('Unrated')

print("Cleaned rating distribution:")
print(df['rating_clean'].value_counts())

Cleaned rating distribution:
rating_clean
Unrated    30395
Name: count, dtype: int64


In [15]:
# Check the actual raw values with repr
for val in df['Rating'].dropna().unique()[:10]:
    print(repr(val))

' Rated T+'
' Rated T'
' ALL AGES'
' A'
' Parental Advisory'
' Marvel Psr'
' No Rating'
' MARVEL PSR'
' T'
' RATED T'


In [16]:
# Strip whitespace first, then map
df['Rating'] = df['Rating'].str.strip()
df['rating_clean'] = df['Rating'].map(rating_map).fillna('Unrated')

print("Cleaned rating distribution:")
print(df['rating_clean'].value_counts())

Cleaned rating distribution:
rating_clean
Unrated              18100
Teen+                 4323
Teen                  3020
All Ages              2071
Parental Advisory     1432
Marvel PSR            1030
Mature/Explicit        329
No Rating               90
Name: count, dtype: int64


In [17]:
# Clean imprint casing
df['imprint_clean'] = df['Imprint'].str.strip().str.title().fillna('Unknown')

# Create modeling subset - only rows with usable ratings
valid_ratings = ['All Ages', 'Teen', 'Teen+',
                 'Parental Advisory', 'Mature/Explicit']
df_model = df[df['rating_clean'].isin(valid_ratings)].copy()

print(f"Full dataset (for EDA/sentiment): {len(df)} rows")
print(f"Modeling dataset (for prediction): {len(df_model)} rows")

Full dataset (for EDA/sentiment): 30395 rows
Modeling dataset (for prediction): 11175 rows


---

## Exploratory Data Analysis

Visualising rating distributions, description lengths by rating, and rating breakdowns across imprints to understand the dataset's structure before applying NLP techniques.



In [18]:
# Description length feature
df['desc_length'] = df['issue_description'].str.len()
df_model['desc_length'] = df_model['issue_description'].str.len()

alt.data_transformers.enable('json')

# Rating distribution
alt.Chart(df_model).mark_bar().encode(
    x=alt.X('rating_clean:N', sort='-y', title='Rating'),
    y=alt.Y('count():Q', title='Count'),
    color=alt.Color('rating_clean:N', legend=None)
).properties(
    title='Rating Distribution (Modeling Dataset)',
    width=400,
    height=300
)

alt.Chart(...)

In [19]:
# Description length by rating
alt.Chart(df_model).mark_boxplot().encode(
    x=alt.X('rating_clean:N', sort='-y', title='Rating'),
    y=alt.Y('desc_length:Q', title='Description Length (chars)'),
    color=alt.Color('rating_clean:N', legend=None)
).properties(
    title='Description Length by Rating',
    width=400,
    height=300
)

alt.Chart(...)

In [20]:
# Top imprints by rating breakdown
top_imprints = df_model['imprint_clean'].value_counts().head(8).index.tolist()
df_imprint = df_model[df_model['imprint_clean'].isin(top_imprints)]

alt.Chart(df_imprint).mark_bar().encode(
    x=alt.X('count():Q', title='Count'),
    y=alt.Y('imprint_clean:N', sort='-x', title='Imprint'),
    color=alt.Color('rating_clean:N', title='Rating')
).properties(
    title='Rating Breakdown by Imprint',
    width=500,
    height=300
)

alt.Chart(...)

--- 

## VADER Implementation 

In [21]:
# Initialise VADER
analyzer = SentimentIntensityAnalyzer()

# Run VADER on all descriptions
vader_compounds = []
vader_positives = []
vader_negatives = []
vader_neutrals = []

for desc in df_model['issue_description']:
    scores = analyzer.polarity_scores(desc)
    vader_compounds.append(scores['compound'])
    vader_positives.append(scores['pos'])
    vader_negatives.append(scores['neg'])
    vader_neutrals.append(scores['neu'])

df_model['vader_compound'] = vader_compounds
df_model['vader_pos'] = vader_positives
df_model['vader_neg'] = vader_negatives
df_model['vader_neu'] = vader_neutrals

# Label sentiment
def classify_vader(score):
    if score >= 0.05:
        return 'Positive'
    elif score <= -0.05:
        return 'Negative'
    else:
        return 'Neutral'


df_model['vader_label'] = df_model['vader_compound'].apply(classify_vader)

print("VADER sentiment distribution:")
print(df_model['vader_label'].value_counts())
print(f"\nMean compound score by rating:")
print(df_model.groupby('rating_clean')['vader_compound'].mean().sort_values())

VADER sentiment distribution:
vader_label
Negative    5850
Positive    4563
Neutral      762
Name: count, dtype: int64

Mean compound score by rating:
rating_clean
Parental Advisory   -0.252528
Mature/Explicit     -0.205105
Teen+               -0.105225
Teen                -0.059852
All Ages             0.076407
Name: vader_compound, dtype: float64


In [22]:
# VADER compound score distribution by rating
alt.Chart(df_model).mark_boxplot().encode(
    x=alt.X('rating_clean:N', sort='-y', title='Rating'),
    y=alt.Y('vader_compound:Q', title='VADER Compound Score'),
    color=alt.Color('rating_clean:N', legend=None)
).properties(
    title='VADER Sentiment by Rating',
    width=400,
    height=300
)

alt.Chart(...)

### VADER Analysis
Applying VADER, a rule based sentiment analyser that scores text using a pre built lexicon of word sentiment values. Unlike classical ML approaches such as logistic regression or SVMs which require labelled training data and manually engineered features like TF-IDF vectors, VADER requires no training, it provides sentiment scores out of the box. However, like TF-IDF, it treats words independently and cannot understand context, word order, or nuance (e.g. it cannot distinguish "not good" from "good"). Results reveal a clear sentiment gradient across ratings, with All Ages descriptions scoring most positive (0.336) and Mature/Explicit lowest (0.106), demonstrating that even promotional marketing language reflects content severity.

--- 
## BERT Implementation 


In [23]:
# BERT Sentiment Analysis
# Using a pre-trained multilingual sentiment model that returns 1-5 star ratings
bert_sentiment = pipeline(
    "sentiment-analysis",
    model="nlptown/bert-base-multilingual-uncased-sentiment",
    truncation=True
)

# Process in batches 
batch_size = 32
bert_results = []
texts = df_model['issue_description'].str[:512].tolist()

for i in range(0, len(texts), batch_size):
    batch = texts[i:i+batch_size]
    results = bert_sentiment(batch)
    bert_results.extend(results)
    if i % 1000 == 0:
        print(f"Processed {i}/{len(texts)} descriptions...")

# Extract star ratings and confidence scores
bert_stars = []
bert_scores = []
for result in bert_results:
    stars = int(result['label'].split()[0])
    bert_stars.append(stars)
    bert_scores.append(result['score'])

df_model['bert_stars'] = bert_stars
df_model['bert_score'] = bert_scores

# Label sentiment from stars
def classify_bert(stars):
    if stars >= 4:
        return 'Positive'
    elif stars <= 2:
        return 'Negative'
    else:
        return 'Neutral'


df_model['bert_label'] = df_model['bert_stars'].apply(classify_bert)

print("BERT sentiment distribution:")
print(df_model['bert_label'].value_counts())
print(f"\nMean BERT stars by rating:")
print(df_model.groupby('rating_clean')['bert_stars'].mean().sort_values())

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Processed 0/11175 descriptions...
Processed 4000/11175 descriptions...
Processed 8000/11175 descriptions...
BERT sentiment distribution:
bert_label
Positive    9499
Negative    1043
Neutral      633
Name: count, dtype: int64

Mean BERT stars by rating:
rating_clean
Mature/Explicit      3.851064
Parental Advisory    3.886872
Teen                 3.998344
Teen+                4.086514
All Ages             4.102366
Name: bert_stars, dtype: float64


In [24]:
# BERT sentiment by rating
alt.Chart(df_model).mark_boxplot().encode(
    x=alt.X('rating_clean:N', sort='-y', title='Rating'),
    y=alt.Y('bert_stars:Q', title='BERT Star Rating (1-5)'),
    color=alt.Color('rating_clean:N', legend=None)
).properties(
    title='BERT Sentiment by Rating',
    width=400,
    height=300
)

alt.Chart(...)

In [25]:
# Compare VADER vs BERT sentiment by rating
vader_summary = df_model.groupby('rating_clean')[
    'vader_compound'].mean().reset_index()
vader_summary['model'] = 'VADER'
vader_summary.columns = ['rating', 'score', 'model']

bert_summary = df_model.groupby('rating_clean')[
    'bert_stars'].mean().reset_index()
# Normalise BERT stars (1-5) to roughly -1 to 1 scale for comparison
bert_summary['bert_stars'] = (bert_summary['bert_stars'] - 3) / 2
bert_summary['model'] = 'BERT'
bert_summary.columns = ['rating', 'score', 'model']

comparison = pd.concat([vader_summary, bert_summary])

alt.Chart(comparison).mark_bar().encode(
    x=alt.X('rating:N', title='Rating'),
    y=alt.Y('score:Q', title='Normalised Sentiment Score'),
    color=alt.Color('model:N', title='Model'),
    xOffset='model:N'
).properties(
    title='VADER vs BERT Sentiment by Rating',
    width=450,
    height=300
)

alt.Chart(...)

## BERT Sentiment Analysis 

Applying a pre-trained BERT model (nlptown/bert-base-multilingual-uncased-sentiment) which scores text on a 1-5 star scale. Unlike VADER's word level lookups, BERT understands context, word order, and nuance, for example, it can recognise that "not good" is negative even though "good" is a positive word. However, results show minimal variation across rating categories, with all ratings clustering around 4-5 stars. This is because the model was trained on product reviews and interprets the promotional marketing tone of comic descriptions as universally positive, regardless of content severity. By contrast, VADER's simpler lexicon based approach inadvertently captured content rating signals by flagging words like "death" and "brutal" as negative. This demonstrates that a more sophisticated model does not always outperform a simpler one, domain fit matters more than model complexity. Both VADER and BERT scores are retained as features for the prediction model, but we expect VADER to carry more predictive weight.

--- 
## HuggingFace Embeddings

In [26]:
# HuggingFace Embeddings
# Generate dense semantic vectors (384 dimensions) for each description
# These capture the full meaning of the text, not just sentiment
model = SentenceTransformer('all-MiniLM-L6-v2')

# Generate embeddings for all descriptions
texts = df_model['issue_description'].tolist()
print(f"Generating embeddings for {len(texts)} descriptions...")
embeddings = model.encode(texts, show_progress_bar=True, batch_size=64)

print(f"Embedding shape: {embeddings.shape}")
print(f"Each description is now a vector of {embeddings.shape[1]} numbers")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Generating embeddings for 11175 descriptions...


Batches:   0%|          | 0/175 [00:00<?, ?it/s]

Embedding shape: (11175, 384)
Each description is now a vector of 384 numbers


In [27]:
# Reduce 384 dimensions to 2 for visualisation using PCA
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
embeddings_2d = pca.fit_transform(embeddings)

df_model['embed_x'] = embeddings_2d[:, 0]
df_model['embed_y'] = embeddings_2d[:, 1]

print(f"Variance explained: {pca.explained_variance_ratio_.sum():.2%}")

# Scatter plot of embeddings coloured by rating
alt.Chart(df_model.sample(3000)).mark_circle(size=20, opacity=0.5).encode(
    x=alt.X('embed_x:Q', title='PCA Component 1'),
    y=alt.Y('embed_y:Q', title='PCA Component 2'),
    color=alt.Color('rating_clean:N', title='Rating')
).properties(
    title='Comic Description Embeddings by Rating (PCA)',
    width=500,
    height=400
)

Variance explained: 7.78%


alt.Chart(...)

## HuggingFace Semantic Embeddings Analysis
Using the sentence transformers library to convert each comic description into a vector of 384 numbers that represent the meaning of the text. Unlike TF-IDF which counts word importance, or VADER which scores positive/negative tone, embeddings capture what the description is actually about, two descriptions with completely different words but similar themes will produce similar vectors. We used PCA to compress these 384 dimensions down to 2 so we could plot them. The scatter plot shows no obvious separation between ratings, meaning the descriptions don't neatly split into groups based on content rating alone. This is expected, comic descriptions are short marketing blurbs that often follow similar structures regardless of rating. However, the full 384-dimensional vectors may still contain useful patterns that our prediction model can pick up, even if we can't see them in a 2D projection.

Note: PCA variance explained was 7.7%, meaning only 7.7% of the information from the original 384 dimensions was retained in the 2D projection. The remaining 92.3% was lost in compression, this is why the scatter plot appears mixed, and why the full embeddings may still contain useful signal that the classifier can exploit.

--- 

## Category Based Analysis

In [28]:
# Category-Based Analysis
# Clean format column
df_model['format_clean'] = df_model['Format'].str.strip(
).str.title().fillna('Unknown')

# Rating distribution by top imprints
top_imprints = df_model['imprint_clean'].value_counts().head(6).index.tolist()
df_imp = df_model[df_model['imprint_clean'].isin(top_imprints)]

alt.Chart(df_imp).mark_bar().encode(
    x=alt.X('count():Q', title='Count'),
    y=alt.Y('imprint_clean:N', sort='-x', title='Imprint'),
    color=alt.Color('rating_clean:N', title='Rating')
).properties(
    title='Rating Distribution by Imprint',
    width=500,
    height=300
)

alt.Chart(...)

In [29]:
# Rating proportions by imprint (normalised)
alt.Chart(df_imp).mark_bar().encode(
    x=alt.X('count():Q', stack='normalize', title='Proportion'),
    y=alt.Y('imprint_clean:N', sort='-x', title='Imprint'),
    color=alt.Color('rating_clean:N', title='Rating')
).properties(
    title='Rating Proportions by Imprint (Normalised)',
    width=500,
    height=300
)

alt.Chart(...)

In [30]:
# Average VADER sentiment by imprint
imprint_sentiment = df_imp.groupby('imprint_clean')[
    'vader_compound'].mean().reset_index()
imprint_sentiment.columns = ['imprint', 'mean_vader']

alt.Chart(imprint_sentiment).mark_bar().encode(
    x=alt.X('imprint:N', sort='-y', title='Imprint'),
    y=alt.Y('mean_vader:Q', title='Mean VADER Compound Score'),
    color=alt.Color('imprint:N', legend=None)
).properties(
    title='Average Sentiment by Imprint',
    width=400,
    height=300
)

alt.Chart(...)

## CategoryBased Analysis
Examining how ratings and sentiment distribute across Marvel's sub brands. The normalised proportion chart confirms that imprints align with their intended audiences, Marvel Adventures is nearly 100% All Ages, while MAX is almost entirely Mature/Explicit. VADER sentiment scores validate this further, showing a clear gradient from Marvel Adventures (the only positively scoring imprint at 0.18) down to MAX (the most negative at -0.28). This acts as a sanity check for our sentiment analysis, the scores align with real editorial intent, confirming that VADER is capturing meaningful signal from the descriptions.

---

## Prediction Model 

In [31]:
# Rating Prediction Model
# Combining all features: embeddings + sentiment scores + description length

# Embeddings (384 features)
X_embeddings = embeddings

# Sentiment and metadata features
sentiment_features = df_model[['vader_compound', 'vader_pos', 'vader_neg', 'vader_neu',
                               'bert_stars', 'desc_length']].values

# Combine all features
X = np.hstack([X_embeddings, sentiment_features])
print(f"Feature matrix shape: {X.shape}")
print(f"  - 384 embedding dimensions")
print(f"  - 6 sentiment/metadata features")
print(f"  - {X.shape[1]} total features")

# Encode target labels
le = LabelEncoder()
y = le.fit_transform(df_model['rating_clean'])
print(f"\nTarget classes: {list(le.classes_)}")
print(f"Target distribution:")
for cls, count in zip(le.classes_, np.bincount(y)):
    print(f"  {cls}: {count}")

Feature matrix shape: (11175, 390)
  - 384 embedding dimensions
  - 6 sentiment/metadata features
  - 390 total features

Target classes: ['All Ages', 'Mature/Explicit', 'Parental Advisory', 'Teen', 'Teen+']
Target distribution:
  All Ages: 2071
  Mature/Explicit: 329
  Parental Advisory: 1432
  Teen: 3020
  Teen+: 4323


In [32]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} rows")
print(f"Test set: {X_test.shape[0]} rows")

# Train Random Forest classifier
rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

# Evaluate
y_pred = rf.predict(X_test)
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

Training set: 8940 rows
Test set: 2235 rows

Classification Report:
                   precision    recall  f1-score   support

         All Ages       0.82      0.22      0.35       414
  Mature/Explicit       1.00      0.15      0.26        66
Parental Advisory       0.79      0.34      0.47       286
             Teen       0.56      0.33      0.42       604
            Teen+       0.49      0.92      0.64       865

         accuracy                           0.54      2235
        macro avg       0.73      0.39      0.43      2235
     weighted avg       0.62      0.54      0.49      2235



In [33]:
# Try with class weight balancing to handle imbalanced classes
rf_balanced = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
    class_weight='balanced'
)
rf_balanced.fit(X_train, y_train)

y_pred_balanced = rf_balanced.predict(X_test)
print("Classification Report (Balanced Weights):")
print(classification_report(y_test, y_pred_balanced, target_names=le.classes_))

Classification Report (Balanced Weights):
                   precision    recall  f1-score   support

         All Ages       0.80      0.20      0.32       414
  Mature/Explicit       0.85      0.26      0.40        66
Parental Advisory       0.82      0.33      0.47       286
             Teen       0.66      0.25      0.37       604
            Teen+       0.47      0.95      0.63       865

         accuracy                           0.52      2235
        macro avg       0.72      0.40      0.44      2235
     weighted avg       0.64      0.52      0.47      2235



In [34]:
# Feature Importance - which features contributed most?
feature_names = [f"embed_{i}" for i in range(384)] + [
    'vader_compound', 'vader_pos', 'vader_neg', 'vader_neu',
    'bert_stars', 'desc_length'
]

importances = rf.feature_importances_

# Get top 20 most important features
top_idx = np.argsort(importances)[-20:]
top_features = pd.DataFrame({
    'feature': [feature_names[i] for i in top_idx],
    'importance': importances[top_idx]
})

alt.Chart(top_features).mark_bar().encode(
    x=alt.X('importance:Q', title='Importance'),
    y=alt.Y('feature:N', sort='-x', title='Feature')
).properties(
    title='Top 20 Most Important Features',
    width=450,
    height=400
)

alt.Chart(...)

## Feature Importance

Analysing which features contributed most to the Random Forest's predictions. The top 20 features were almost entirely embedding dimensions, with description length being the only non-embedding feature to appear. Notably, none of the VADER or BERT sentiment scores made the top 20. This tells us that the semantic content of a description (what it's actually about) is far more predictive of its rating than whether it sounds positive or negative. This makes intuitive sense: a description mentioning "family friendly adventure" versus "violent underworld crime" differs in meaning, not just tone. Sentiment is just one narrow aspect of language, while the embeddings capture a much broader picture including themes, character types, and writing style.

---

## LLM Analysis

In [49]:
# LLM-Based Classification
# Using Claude to directly classify comic descriptions into ratings
client = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

# Take a stratified sample of 25 descriptions (5 per rating)
samples_list = []
for rating in df_model['rating_clean'].unique():
    rating_df = df_model[df_model['rating_clean'] == rating]
    n = min(5, len(rating_df))
    rating_sample = rating_df.sample(n=n, random_state=42)
    samples_list.append(rating_sample)

sample = pd.concat(samples_list).reset_index(drop=True)

# Format descriptions for the prompt
descriptions_text = ""
for i, row in sample.iterrows():
    descriptions_text += f"Description {i+1}: {row['issue_description'][:300]}\n\n"

prompt = f"""You are analysing Marvel comic book descriptions to predict their content rating.
The possible ratings are: All Ages, Teen, Teen+, Parental Advisory, Mature/Explicit.

For each description below, predict the most likely rating.
Return ONLY a numbered list in this exact format, nothing else:
1. Rating
2. Rating
etc.

{descriptions_text}"""

response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=2048,
    messages=[
        {"role": "user", "content": prompt}
    ]
)

llm_output = response.content[0].text
print(llm_output)

1. All Ages
2. Teen
3. Teen+
4. Teen
5. Teen
6. Teen
7. All Ages
8. Teen
9. Teen
10. Teen+
11. All Ages
12. All Ages
13. All Ages
14. All Ages
15. All Ages
16. Teen+
17. Teen+
18. Teen+
19. Teen+
20. Teen
21. Mature/Explicit
22. Mature/Explicit
23. Teen+
24. Teen+
25. Mature/Explicit


In [50]:
# Parse LLM predictions and compare to actual
llm_lines = llm_output.strip().split("\n")
llm_predictions = []
for line in llm_lines:
    rating = line.split(". ", 1)[1].strip()
    llm_predictions.append(rating)

actual_ratings = sample['rating_clean'].tolist()

correct = 0
print(f"{'#':<4} {'Actual':<22} {'LLM Predicted':<22} {'Match'}")
print("-" * 60)
for i in range(len(actual_ratings)):
    match = actual_ratings[i] == llm_predictions[i]
    if match:
        correct += 1
    symbol = "✓" if match else "✗"
    print(
        f"{i+1:<4} {actual_ratings[i]:<22} {llm_predictions[i]:<22} {symbol}")

print(
    f"\nLLM Accuracy: {correct}/{len(actual_ratings)} ({correct/len(actual_ratings):.0%})")
print(f"Random Forest Accuracy: 54%")

#    Actual                 LLM Predicted          Match
------------------------------------------------------------
1    Teen+                  All Ages               ✗
2    Teen+                  Teen                   ✗
3    Teen+                  Teen+                  ✓
4    Teen+                  Teen                   ✗
5    Teen+                  Teen                   ✗
6    Teen                   Teen                   ✓
7    Teen                   All Ages               ✗
8    Teen                   Teen                   ✓
9    Teen                   Teen                   ✓
10   Teen                   Teen+                  ✗
11   All Ages               All Ages               ✓
12   All Ages               All Ages               ✓
13   All Ages               All Ages               ✓
14   All Ages               All Ages               ✓
15   All Ages               All Ages               ✓
16   Parental Advisory      Teen+                  ✗
17   Parental Advisory      Teen+ 

## LLM-Based Classification
Using Claude as a zero-shot classifier,  giving it comic descriptions with no training data and asking it to predict ratings directly. The LLM achieved 52% accuracy on a 25-description sample, comparable to the trained Random Forest (54%). Performance was strongest at the extremes: All Ages was classified perfectly (5/5) and Mature/Explicit scored well (3/5), as these ratings have the most distinctive language. The middle categories (Teen, Teen+, Parental Advisory) proved difficult, with Parental Advisory scoring 0/5. This confirms that the boundary between these ratings is genuinely subjective, even a model with deep language understanding cannot reliably distinguish them from short marketing descriptions alone. The key advantage of the LLM approach is that it required no training, no feature engineering, and can explain its reasoning, but it's slower and more expensive to run at scale.

---

## Conclusion
This project explored whether Marvel comic book descriptions could predict content ratings using a range of NLP techniques. VADER sentiment analysis revealed a clear gradient across ratings, with All Ages descriptions scoring most positive (0.336) and Mature/Explicit most negative (0.106). This was validated by category analysis, where imprint level sentiment followed the expected pattern, Marvel Adventures scored the most positive (0.18) while MAX scored the most negative (-0.28), confirming that VADER was capturing real editorial intent from the descriptions.

BERT sentiment, while more sophisticated in its understanding of language, showed minimal variation across ratings due to domain mismatch, the model was trained on product reviews and interpreted all promotional marketing copy as positive regardless of content severity. This demonstrated that a more complex model does not always outperform a simpler one, domain fit matters more than model complexity.

HuggingFace semantic embeddings captured the richest representation of the descriptions, and dominated the top 20 most important features in the prediction model, with description length being the only non embedding feature to appear. This showed that the overall meaning of a description is far more informative for predicting ratings than sentiment scores alone.

The Random Forest classifier achieved 54% accuracy across five classes, well above the 20% random baseline but limited by heavy class imbalance and the promotional nature of the descriptions. A zero shot LLM approach achieved a comparable 52% on a sample of 25 descriptions with no training or feature engineering. Both approaches performed best at the extremes, All Ages and Mature/Explicit have distinctive language, but struggled with the middle categories (Teen, Teen+, Parental Advisory), confirming that these boundaries are genuinely subjective and difficult to distinguish from short marketing copy alone.